In [40]:
i = 1 
k = 4
print(i+k)

5


In [ ]:
!pip install transformers==4.38.0 -q



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 34.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.


In [41]:
# 현재 런타임 확인

import subprocess
import os
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except:
    print("GPU 없음 - CPU 런타임")

print("RAM:", os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3), "GB")

Mon Apr  6 03:24:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             27W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
# vscode에서 코랩 연결하는 방법
# 1. Extention 들어가서 Google Colab 다운
# 2. ipynb 파일 안에서 맨위 상단 오른쪽 Python 3.12혹은 버전 명시되어 있는 부분 클릭
# 3. 상단에 select another kernel 클릭
# 4. Colab 클릭
# 5. 연결 누르고 colab pro 구글 계정 연결하고 
# 6. 오른쪽 상단에 이제 Python 3(ipykernel)확인

# colab 안에서도 repository 클론 떠줘야함.
# !git clone https://github.com/yeseul-kim01/2026-Text2Graph.git


In [56]:
# !git clone https://github.com/yeseul-kim01/2026-Text2Graph.git
# %cd 2026-Text2Graph
!git pull

# 라이브러리 install
# !pip install torch transformers opt-einsum ujson tqdm wandb 

# 모델 dreeam 폴더로 이동
# %cd /content/2026-Text2Graph/project1/models/dreeam

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 9 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 3.81 KiB | 1.27 MiB/s, done.
From https://github.com/yeseul-kim01/2026-Text2Graph
   369febc..d420b67  dev        -> origin/dev
   c7c5683..8ef57d9  jaeyoon    -> origin/jaeyoon
Updating 369febc..d420b67
Fast-forward
 project1/gpuRuntime.ipynb            |   4 +-
 project1/models/dreeam/evaluation.py |   4 +-
 project1/runtimeCheck.ipynb          | 488 ++++++++++++++++++-----------------
 3 files changed, 258 insertions(+), 238 deletions(-)


In [44]:
#확인
os.listdir('/content/2026-Text2Graph/project1')
print("데이터:", os.listdir('/content/2026-Text2Graph/project1/data'))
print("모델:", os.listdir('/content/2026-Text2Graph/project1/models/dreeam'))


데이터: ['dev.json', 'rel_info.json', 'train_annotated.json', 'test.json']
모델: ['flow.png', 'model.py', 'dataset', 'scripts', 'utils.py', 'losses.py', 'run.py', '__pycache__', 'README.md', 'evaluation.py', 'prepro.py', 'meta', 'requirements.txt', 'heatmap_visualization.ipynb', 'args.py', 'long_seq.py', 'LICENSE']


In [45]:
#데이터 연결 (DREEAM이 요구하는 폴더 구조 만들기)
!mkdir -p dataset/docred
!ln -s /content/2026-Text2Graph/project1/data/train_annotated.json dataset/docred/train_annotated.json
!ln -s /content/2026-Text2Graph/project1/data/dev.json dataset/docred/dev.json

# 확인
!ls dataset/docred/

#dev.json  train_annotated.json 나오면성공

ln: failed to create symbolic link 'dataset/docred/train_annotated.json': File exists
ln: failed to create symbolic link 'dataset/docred/dev.json': File exists
dev.json  train_annotated.json


In [57]:
!WANDB_MODE=disabled python run.py \
    --do_train \
    --data_dir ./dataset/docred \
    --transformer_type bert \
    --model_name_or_path bert-base-uncased \
    --train_file train_annotated.json \
    --dev_file dev.json \
    --save_path ./checkpoint \
    --train_batch_size 4 \
    --test_batch_size 4 \
    --num_train_epochs 30 \
    --lr_transformer 5e-5 \
    --lr_added 1e-4 \
    --num_class 97 \
    --num_labels 4 \
    --evi_lambda 0.1 \
    --seed 42 \
    --warmup_ratio 0.06 \
    --max_grad_norm 1.0

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading examples: 100% 3053/3053 [00:35<00:00, 86.45it/s] 
# of documents 3053.
# of positive examples 35615.
# of negative examples 1163035.
Loading examples: 100% 998/998 [00:12<00:00, 78.82it/s] 
# of documents 998.
# of positive examples 11470.
# of negative examples 384102.
/content/2026-Text2Graph/project1/models/dreeam/run.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Total steps: 22890
Warmup steps: 1373
Train epoch:   0% 0/30 [00:00<?, ?it/s]
Evaluating batches:   0% 0/250 [00:00<?, ?it/s]
Evaluating batches:   0% 1/250 [00:00<00:29,  8.56it/s]
Evaluating batches:   1% 2/250 [00:00<00:34, 

In [ ]:
# 이러한 오류가 발생시!!!!!!!!! 최신 transformers에서 AdamW 위치가 바뀌어서 그럼.

# Traceback (most recent call last):
#   File "/content/2026-Text2Graph/project1/models/dreeam/run.py", line 11, in <module>
#     from transformers.optimization import AdamW, get_linear_schedule_with_warmup
# ImportError: cannot import name 'AdamW' from 'transformers.optimization' (/usr/local/lib/python3.12/dist-packages/transformers/optimization.py)

# 아래 명령어 실행
!sed -i 's/from transformers.optimization import AdamW, get_linear_schedule_with_warmup/from torch.optim import AdamW\nfrom transformers import get_linear_schedule_with_warmup/' run.py


In [ ]:
# 이러한 오류가 발생시!!!!!!!!! meta/rel2id.json 파일이 없어서 그럼.

!mkdir -p meta
import json

# rel_info.json에서 relation 목록 가져와서 rel2id 생성
rel_info = json.load(open('/content/2026-Text2Graph/project1/data/rel_info.json', 'r'))

rel2id = {"Na": 0}  # 관계 없음 = 0번
for i, rel in enumerate(rel_info.keys()):
    rel2id[rel] = i + 1

json.dump(rel2id, open('meta/rel2id.json', 'w'))

# 확인
print(f"총 {len(rel2id)}개 관계")
print(dict(list(rel2id.items())[:5]))

총 97개 관계
{'Na': 0, 'P6': 1, 'P17': 2, 'P19': 3, 'P20': 4}
